# text-orientation-classifier. Насибуллин Никита

Определение ориентации перевернутых изображений текста: 0° или 180°.
Бинарная классификация, на выходе вероятность `p_180`.

Метрика соревнования: Brier Score, `Score = 1 - Brier`.

## Подход

### Данные

Два синтетических OCR-датасета: русский (50k) ,английский (50k)
и 5k реальных английских изображений из IIIT-5K как небольшой
домен-сдвиг в сторону реальных фото. Итого 105k исходных изображений.

Ключевой момент: train/val делается **на уровне исходных
изображений**, до генерации пар 0°/180°. Для каждого исходника
создаются два примера — оригинал (label 0) и поворот на 180°
(label 1). Если сплитить после генерации пар, оригинал попадёт
в train, а его поворот в val, и модель увидит один и тот же текст
в обеих выборках. Получается 84k источников в train (168k примеров)
и 21k в val (42k примеров).

### Препроцессинг

Изображения имеют очень разное соотношение сторон, поэтому обычный
`Resize((224, 224))` искажал бы текст и убивал признаки ориентации.
Используется resize с сохранением пропорций и белым padding
до квадрата 224×224.

### Модель

MobileNetV3-Small с весами ImageNet. 

- компактность (~1.5M параметров) и скорость на CPU — соответствует
  требованию соревнования о компактной модели;
- предобучение на ImageNet даёт хорошие низкоуровневые признаки
  (края, текстуры, контуры букв), что важно при небольшом объёме
  целевых данных относительно размера модели;

Последний слой классификатора заменён с `Linear(1024, 1000)`
на `Linear(1024, 1)` — один логит под бинарную задачу. Обучается
вся модель целиком, без заморозки слоёв.

### Функция потерь

`BCEWithLogitsLoss`. Эта функция потерь объединяет sigmoid и бинарную
кросс-энтропию в одной численно устойчивой операции:

```
loss = -[y * log(sigmoid(z)) + (1 - y) * log(1 - sigmoid(z))]
```



### Оптимизатор

`AdamW`, `lr=1e-4`. AdamW
Батч 32, CPU-only, `num_workers=0`.

### Валидация

Основная метрика — Brier Score, как в соревновании:

```
Brier = mean((p - y)^2)
Score = 1 - Brier
```

Accuracy считается для справки


In [2]:
# Проверяем версии Python, PyTorch и Torchvision.
# Это важно для воспроизводимости: обучение CPU-only,
# поэтому фиксируем, что запускается именно эта связка.
import sys
import torch
import torchvision

print("Python:", sys.version)
print("Executable:", sys.executable)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

Python: 3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]
Executable: C:\Users\Никита\AppData\Local\Python\pythoncore-3.14-64\python.exe
PyTorch: 2.14.0+cpu
Torchvision: 0.29.0+cpu


In [3]:
from pathlib import Path
import random
# Функция рекурсивно собирает все изображения
# с нужными расширениями из папок.
def get_images(folder):
    folder = Path(folder)

    extensions = {".png", ".jpg", ".jpeg", ".webp"}

    return [
        p for p in folder.rglob("*")
        if p.suffix.lower() in extensions
    ]


eng_images = get_images("eng")
rus_images = get_images("rus")

print("ENG:", len(eng_images))
print("RUS:", len(rus_images))

ENG: 76516
RUS: 500000


In [4]:
# Берём сбалансированную подвыборку: ровно по 50 000 изображений
# каждого языка. random.seed фиксирует выбор, чтобы результат воспроизводился.
random.seed(42)

eng_images = random.sample(eng_images, 50_000)
rus_images = random.sample(rus_images, 50_000)

print("ENG:", len(eng_images))
print("RUS:", len(rus_images))

ENG: 50000
RUS: 50000


In [115]:
# IIIT-5K Word Dataset — реальные фотографии вывесок и сцен с текстом
eng_pro_images = get_images('eng_pro')

In [31]:
from PIL import Image
# Приводим изображение к квадрату size×size с сохранением
# пропорций: масштабируем по меньшей стороне и докладываем
# белыми полями до квадрата.
def resize_with_padding(image, size=224):

    width, height = image.size

    scale = min(size / width, size / height)

    new_width = round(width * scale)
    new_height = round(height * scale)

    image = image.resize(
        (new_width, new_height),
        Image.Resampling.LANCZOS
    )

    canvas = Image.new(
        "RGB",
        (size, size),
        "white"
    )

    x = (size - new_width) // 2
    y = (size - new_height) // 2

    canvas.paste(image, (x, y))

    return canvas

In [37]:
class OrientationDataset(Dataset):
     """
    Датасет для бинарной классификации ориентации текста.

    Для каждого исходного изображения генерируются ДВА примера:
        idx чётный  -> оригинал, label=0
        idx нечётный -> rotate(180°), label=1

    Поэтому len(dataset) = 2 * len(image_paths).
    """
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths) * 2

    def __getitem__(self, idx):

        image_idx = idx // 2
        label = idx % 2

        image = Image.open(
            self.image_paths[image_idx]
        ).convert("RGB")

        if label == 1:
            image = image.rotate(180)

        image = resize_with_padding(image)

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32)

In [47]:
import torch.nn as nn

model.classifier[3] = nn.Linear(1024, 1)
print(model.classifier)

Sequential(
  (0): Linear(in_features=576, out_features=1024, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=True)
  (3): Linear(in_features=1024, out_features=1, bias=True)
)


In [49]:
criterion = torch.nn.BCEWithLogitsLoss()

In [52]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)


In [134]:
import random

random.seed(42)

eng_selected = random.sample(eng_images, 50_000)
rus_selected = random.sample(rus_images, 50_000)

all_images = eng_selected + rus_selected + eng_pro_images

print(len(eng_selected))
print(len(rus_selected))
print(len(all_images))
random.shuffle(all_images)

50000
50000
105000


In [135]:
# Split 80/20 на уровне исходных изображений
# Это гарантирует, что оригинал и его поворот не окажутся в разных выборках 
split = int(0.8 * len(all_images))

train_images = all_images[:split]
val_images = all_images[split:]

print("train:", len(train_images))
print("val:", len(val_images))

train: 84000
val: 21000


In [136]:
from torchvision import transforms

# Финальный transform для обучения и валидации:
# (совпадает со статистикой, на которой обучался MobileNet)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [137]:
train_dataset = OrientationDataset(
    train_images,
    transform=transform
)

val_dataset = OrientationDataset(
    val_images,
    transform=transform
)
# train: 84000 источников × 2 = 168000 примеров
# val:   21000 источников × 2 =  42000 примеров

print(len(train_dataset))
print(len(val_dataset))

168000
42000


In [140]:
from torch.utils.data import DataLoader
# batch_size=32.
# num_workers=0, т.к. обучение CPU.
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

In [90]:
#Создание модели и изменение ее последнего слоя с 1024ех фичей - до одной

import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

weights = MobileNet_V3_Small_Weights.DEFAULT

model = mobilenet_v3_small(weights=weights)

model.classifier[3] = nn.Linear(1024, 1)

In [91]:
print(model.classifier)

Sequential(
  (0): Linear(in_features=576, out_features=1024, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=True)
  (3): Linear(in_features=1024, out_features=1, bias=True)
)


In [92]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)

In [ ]:
# Одна эпоха обучения. Предыдущие эпохи запускались в отдельных
# прогонах ноутбука, модель между ними не переинициализировалась -
# веса продолжали обновляться.
model.train()

for batch_idx, (images, labels) in enumerate(train_loader):

    optimizer.zero_grad()

    outputs = model(images)

    loss = criterion(
        outputs.squeeze(1),
        labels
    )

    loss.backward()
    optimizer.step()

    if batch_idx % 100 == 0:
        print(
            f"batch {batch_idx}/{len(train_loader)}, "
            f"loss={loss.item():.4f}"
        )

batch 0/5250, loss=0.0639
batch 100/5250, loss=0.0185
batch 200/5250, loss=0.0117
batch 300/5250, loss=0.0165
batch 400/5250, loss=0.0321
batch 500/5250, loss=0.0131
batch 600/5250, loss=0.1287
batch 700/5250, loss=0.0273
batch 800/5250, loss=0.0511
batch 900/5250, loss=0.1109
batch 1000/5250, loss=0.0877
batch 1100/5250, loss=0.0461
batch 1200/5250, loss=0.1773
batch 1300/5250, loss=0.0829
batch 1400/5250, loss=0.0260
batch 1500/5250, loss=0.0282
batch 1600/5250, loss=0.0273
batch 1700/5250, loss=0.0409
batch 1800/5250, loss=0.0161
batch 1900/5250, loss=0.0482
batch 2000/5250, loss=0.1025
batch 2100/5250, loss=0.0232
batch 2200/5250, loss=0.0379
batch 2300/5250, loss=0.0297
batch 2400/5250, loss=0.0409
batch 2500/5250, loss=0.0271
batch 2600/5250, loss=0.1385
batch 2700/5250, loss=0.0379
batch 2800/5250, loss=0.1812
batch 2900/5250, loss=0.0285
batch 3000/5250, loss=0.0372
batch 3100/5250, loss=0.0714
batch 3200/5250, loss=0.0658
batch 3300/5250, loss=0.0355
batch 3400/5250, loss=0.21

In [ ]:
torch.save(model.state_dict(), "mobilenet_epoch5.pth")

In [100]:
model.eval()

total_brier = 0
total_correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        logits = model(images).squeeze(1)
        probs = torch.sigmoid(logits)

        total_brier += ((probs - labels) ** 2).sum().item()

        total_correct += (
            (probs >= 0.5) == (labels == 1)
        ).sum().item()

        total += labels.numel()

brier = total_brier / total
score = 1 - brier
accuracy = total_correct / total

print("Brier:", brier)
print("Score:", score)
print("Accuracy:", accuracy)

Brier: 0.028185708562022772
Score: 0.9718142914379773
Accuracy: 0.9619


In [ ]:

# Папка с тестовыми изображениями
test_dir = Path("images")

test_images = get_images(test_dir)

print(f"Количество тестовых изображений: {len(test_images)}")

In [ ]:
class TestDataset(Dataset):
    """
    Датасет для инференса на тесте.
    Отличие от OrientationDataset:
      - не генерирует пары, просто отдаёт одно изображение
      - возвращает image_id
    """
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]

        image = Image.open(path).convert("RGB")
        image = resize_with_padding(image)

        if self.transform:
            image = self.transform(image)

        # test_00000.png -> test_00000
        image_id = path.stem

        return image, image_id

In [ ]:
test_dataset = TestDataset(
    test_images,
    transform=transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

In [ ]:
model.eval()

predictions = []
image_ids = []

with torch.no_grad():
    for images, ids in test_loader:
        logits = model(images).squeeze(1)
        probs = torch.sigmoid(logits)

        predictions.extend(probs.cpu().numpy())
        image_ids.extend(ids)

In [ ]:
# Формируем submission в формате соревнования: image_id, p_180.
import pandas as pd
submission = pd.DataFrame({
    "image_id": image_ids,
    "p_180": predictions
})

submission.to_csv(
    "submission.csv",
    index=False
)

submission.head(20)